In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)


In [ ]:
np.random.seed(42)
n_patients = 500

data = pd.DataFrame({
    'age': np.random.randint(18, 90, n_patients),
    'previous_visits': np.random.poisson(2, n_patients),
    'heart_rate': np.random.normal(80, 12, n_patients).round(1),
    'systolic_bp': np.random.normal(130, 18, n_patients).round(1),
    'diagnosis': np.random.choice(
        ['Diabetes', 'Heart Disease', 'Infection', 'Injury'],
        n_patients
    )
})

# Create a simple probability of readmission for demonstration.
# This is artificial data and has no medical meaning.
risk_score = (
    -3.0
    + 0.025 * data['age']
    + 0.30 * data['previous_visits']
    + 0.35 * (data['diagnosis'] == 'Heart Disease')
    + 0.20 * (data['diagnosis'] == 'Diabetes')
)

probability = 1 / (1 + np.exp(-risk_score))
data['readmitted_30_days'] = np.random.binomial(1, probability)

data.head()


In [ ]:
print('Dataset shape:', data.shape)
print('\nData types:')
print(data.dtypes)
print('\nMissing values:')
print(data.isnull().sum())
print('\nTarget distribution:')
print(data['readmitted_30_days'].value_counts())


In [ ]:
X = data.drop(columns=['readmitted_30_days'])
y = data['readmitted_30_days']

numeric_features = [
    'age',
    'previous_visits',
    'heart_rate',
    'systolic_bp'
]

categorical_features = ['diagnosis']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])


In [ ]:
model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=1.0,
        max_iter=1000,
        random_state=42
    ))
])

# Train the complete pipeline.
model.fit(X_train, y_train)


In [ ]:
y_probability = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print('First five predicted probabilities:', y_probability[:5])
print('First five class predictions:', y_pred[:5])


In [ ]:
auc_score = roc_auc_score(y_test, y_probability)
print(f'ROC-AUC: {auc_score:.3f}')

false_positive_rate, true_positive_rate, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(7, 5))
plt.plot(false_positive_rate, true_positive_rate, label=f'ROC-AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate / Recall')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No readmission', 'Readmission']
)
display.plot()
plt.title('Confusion Matrix')
plt.show()
